# Investigating Genes found by the Neural Networks

### NCBI Blast then Gene Search

In [1]:
from pathlib import Path
import os

# If this script is in /project/scripts/script.py, .parent.parent gets /project/
root = "/home/projects/s215045/PredictPhagePPI/"
raw_data_path = os.path.join(root, "raw_data/")
data_prod_path = os.path.join(root, "data_prod/")

path_to_nn_runs = os.path.join(root, "nn_runs/")

In [ ]:
# Read nn_runs
nn_run = "inv_attr_genes_run7"
# if nn_run not in os.listdir(path_to_nn_runs):
#     raise ValueError("No run found")

def extract_kmer_line(file_path):
    # Check if filepath exists
    try:
        os.path.exists(file_path)
    except FileExistsError as e:
        print("File path doesn't exist")
    
    with open(file_path, "r") as logfile:
        for line in logfile:
            if "Top 10 decoded kmers:" in line:
                return line

def clean_kmer_line(kmer_line):
    """Clean the line containing kmers, from a messy string with noise, to a list with only decoded kmers"""
    kmers_string = kmer_line.split(":")[-1].strip()
    return kmers_string.strip("[]").replace("'", "").split(", ")

kmer_line = extract_kmer_line(path_to_nn_runs+nn_run+"/log_run7.txt")
print(clean_kmer_line(kmer_line))

In [ ]:
import time
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import Entrez, SeqIO

# 1. Configuration
Entrez.email = "s215045@student.dtu.dk"  # Replace with your email
kmers = ['CACAGCAAGCAA', 'AGAGAAGAAAGT', 'AATCACTGTCAA', 'TTCGCGTCAGAA', 'ATGACATACCAT', 'GAACAATGAGCC', 'AAGTTGAATTTG', 'ATGAAGCTGGTT', 'GGGTAAATATCC', 'AAGAGTGCTTGA']

def search_and_annotate_kmers(kmer_list, outfile:str = root+"logs/NCBI_gene_search.txt", acc_num:int = 3, tax_origin:str  = "txid38018[orgn]", ncbi_program:str = "blastn", ncbi_db:str = "core_nt"):
    """
    Blasts each of the kmers against NCBI, for related species (accessions), then searches its genes for the kmer along with possible functionalities
    """
    with open(outfile, "w") as logfile:
        print(f"Starting BLAST for {len(kmer_list)} kmers against Viral Database...", file=logfile)
        
        # We combine kmers into one FASTA-style string to save API calls
        fasta_query = "\n".join([f">kmer_{i}\n{k}" for i, k in enumerate(kmer_list)])
        
        try:
            # qblast parameters for short sequences:
            # - program: blastn
            # - database: nt (nucleotide)
            # - entrez_query: Restrict to Viruses
            # - word_size: 7 (minimum for blastn)
            # - expect: 1000 (higher to catch short hits)
            result_handle = NCBIWWW.qblast(
                program=ncbi_program, 
                database=ncbi_db, 
                sequence=fasta_query,
                entrez_query=tax_origin, #12333
                word_size=7,
                expect=1000,
                short_query=True
            )
            
            blast_records = NCBIXML.parse(result_handle)
            
            for record in blast_records:
                kmer_seq = kmer_list[int(record.query.split('_')[1])]
                print(f"\n--- Results for Kmer: {kmer_seq} ---", file=logfile)
                
                if not record.alignments:
                    print("No significant phage hits found.", file=logfile)
                    continue

                # Check the top acc_num hits for functional relevance
                for alignment in record.alignments[:acc_num]:
                    accession = alignment.accession
                    hit_def = alignment.title
                    
                    # Fetch GenBank record to find the specific gene overlapping the hit
                    print(f"Checking Gene in Hit: {accession} ({hit_def[:50]}...)", file=logfile)
                    
                    # We fetch the specific region of the hit to save bandwidth
                    hsp = alignment.hsps[0]
                    start, end = min(hsp.sbjct_start, hsp.sbjct_end), max(hsp.sbjct_start, hsp.sbjct_end)
                    
                    try:
                        handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")
                        genbank_rec = SeqIO.read(handle, "genbank")
                        handle.close()
                        
                        found_gene = False
                        for feature in genbank_rec.features:
                            if feature.type == "CDS":
                                # Check if the kmer location overlaps with this gene
                                if start >= feature.location.start and end <= feature.location.end:
                                    product = feature.qualifiers.get('product', ['Unknown'])[0]
                                    gene = feature.qualifiers.get('gene', ['N/A'])[0]
                                    print(f"  [MATCH] Found in Gene: {gene} | Function: {product}", file=logfile)
                                    found_gene = True
                                    break
                        if not found_gene:
                            print("  [INFO] Hit is in an intergenic/non-coding region.", file=logfile)
                            
                    except Exception as e:
                        print(f"  [ERROR] Could not fetch details for {accession}: {e}", file=logfile)
                    
                    time.sleep(1) # Be nice to NCBI servers

        except Exception as e:
            print(f"BLAST search failed: {e}", file=logfile)

# Run the search
search_and_annotate_kmers(kmers, path_to_nn_runs+nn_run+"NCBI_gene_search.txt")

Starting BLAST for 10 kmers against Viral Database...

--- Results for Kmer: CACAGCAAGCAA ---
Checking Gene in Hit: OP074436 (gi|2295220018|gb|OP074436.1| MAG: Bacteriophage sp...)
  [MATCH] Found in Gene: N/A | Function: minor tail protein
Checking Gene in Hit: OP075687 (gi|2295372038|gb|OP075687.1| MAG: Bacteriophage sp...)
  [MATCH] Found in Gene: N/A | Function: Macoilin family protein
Checking Gene in Hit: OP075746 (gi|2295399217|gb|OP075746.1| MAG: Bacteriophage sp...)
  [MATCH] Found in Gene: N/A | Function: minor tail protein

--- Results for Kmer: AGAGAAGAAAGT ---
Checking Gene in Hit: OP075745 (gi|2295399102|gb|OP075745.1| MAG: Bacteriophage sp...)
  [MATCH] Found in Gene: N/A | Function: replisome organizer
Checking Gene in Hit: OP075395 (gi|2295356719|gb|OP075395.1| MAG: Bacteriophage sp...)
  [MATCH] Found in Gene: N/A | Function: hypothetical protein
Checking Gene in Hit: OP072956 (gi|2294090896|gb|OP072956.1| MAG: Bacteriophage sp...)
  [MATCH] Found in Gene: N/A | Funct